# CS521 MP2 Part 3 — Colab T4 Experiments

Select **Runtime → Change runtime type → T4 GPU**, then run the cells in order.

This notebook compares PyTorch eager, Inductor, JAX JIT, and a custom CUDA implementation. It checks correctness, records host compilation times and GPU kernel traces, and plots performance as input and filter sizes vary. Additional instructions are available in `COLAB_PART3.md` in the same directory.

The assignment requires using the PyTorch, JAX, and cuDNN versions provided by Colab. **Do not upgrade them with pip.** All experimental measurements are generated when this notebook runs on Colab.


## 1. Upload the GPU source directory

Run `zip -r /tmp/mp2_gpu.zip gpu` from the repository root on your local machine. Run the following cell and select the ZIP file.

If you have already cloned your own repository containing the updated sources and experiment scripts, skip this cell and run `%cd /content/<repo>/gpu` manually. The course starter repository does not include the fixes and experiment scripts used by this notebook.


In [ ]:
from google.colab import files
import io, zipfile
from pathlib import Path
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Select one ZIP file containing the gpu/ directory.')
name, contents = next(iter(uploaded.items()))
with zipfile.ZipFile(io.BytesIO(contents)) as archive:
    archive.extractall('/content')
assert Path('/content/gpu/benchmark_colab.py').exists(), 'The ZIP must contain the updated gpu/ directory.'
%cd /content/gpu


## 2. Check the T4 GPU, CUDA, and JAX backend

If an environment check fails, confirm the GPU runtime type and reconnect to a clean T4 runtime. Resolve the failure before collecting GPU measurements.


In [ ]:
import os, subprocess, sys
from pathlib import Path
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['nvcc', '--version'], check=True)
import torch, jax
assert torch.cuda.is_available(), 'Select a GPU runtime.'
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('JAX:', jax.__version__, 'backend:', jax.default_backend(), 'devices:', jax.devices())
assert jax.default_backend() == 'gpu'
assert 'T4' in torch.cuda.get_device_name(0), 'The assignment requires a T4 GPU.'
for filename in ('myconv.py', 'myconv_jax.py', 'myconv_kernel.cu', 'benchmark_colab.py', 'trace_utils.py', 'plot_colab_results.py'):
    assert Path(filename).is_file(), filename


## 3. Smoke test: one configuration for each implementation

A successful run prints `All 4 experiments passed`. If a test fails, inspect the `.log` files and `failures.json` in the output directory, then resolve the failure before running the full experiment.

The script refuses to overwrite existing output directories. Choose a new directory name when repeating an experiment. Compilation and profiling run in separate worker processes for each configuration and implementation.


In [ ]:
smoke_dir = 'results_smoke'
subprocess.run([sys.executable, 'benchmark_colab.py', '--output', smoke_dir,
                '--sizes', '19', '--filters', '3', '--warmup', '3',
                '--repeats', '5', '--profile-calls', '3'], check=True)


## 4. Full experiment: 9 shapes × 4 implementations

The default configuration varies `H=W ∈ {19,33,65}` and `KH=KW ∈ {3,5,7}`, with N=2, C_in=3, C_out=8, stride=1, padding=1, FP32, and zero bias. Each experiment uses 10 warmup calls, 30 synchronized wall-time measurements, and 5 profiled calls.

Each experiment uses a fresh process and compilation cache to measure cold compilation. The full run may take a while. Avoid running other GPU cells during measurement. Each experiment writes a log; if an experiment fails, the script preserves completed results and exits with an error after processing the remaining experiments.


In [ ]:
results_dir = 'results_full'
subprocess.run([sys.executable, 'benchmark_colab.py', '--output', results_dir], check=True)


## 5. Inspect the measurements and plots

`gpu_span_mean_us` is the average interval from the first GPU kernel's start to the last GPU kernel's completion within an inference. `gpu_kernel_sum_us` is the sum of GPU kernel durations per inference. `steady_wall_median_us` is the median synchronized end-to-end wall time measured without the profiler.

The scope of `host_compile_s` differs across implementations: eager=N/A; Inductor=backend compilation, excluding Dynamo graph capture; JAX=lowering plus XLA compilation; CUDA=extension build, link, and load. `first_call_s` is reported separately and should not be treated uniformly as compilation time. See `COLAB_PART3.md` for the detailed timing definitions.


In [ ]:
import pandas as pd
from IPython.display import display, Image
summary = pd.read_csv(Path(results_dir) / 'summary.csv')
display(summary[['variant', 'H', 'KH', 'correct', 'max_abs_error', 'host_compile_s',
                 'first_call_s', 'gpu_kernel_sum_us', 'gpu_span_mean_us',
                 'kernels_per_call', 'steady_wall_median_us', 'trace_source']])
for filename in ('gpu_span_by_input.png', 'gpu_span_by_filter.png',
                 'gpu_kernels_by_input.png', 'wall_by_input.png', 'compile_by_input.png'):
    display(Image(filename=str(Path(results_dir) / filename)))


## 6. Inspect traces in Perfetto and prepare the report

Open https://ui.perfetto.dev/ and load the four `torch_trace.json` files for the same shape. JAX also exports `jax_native.json.gz`. If `trace_source` indicates that PyTorch Profiler did not capture JAX GPU kernels, use the native trace and state its source in the report.

Search the CPU/GPU tracks for `eager_conv_0`, `inductor_conv_0`, `jax_conv_0`, and `cuda_conv_0`. Capture screenshots and inspect kernel names, counts, and durations. Use the Inductor FX graph and generated code, along with JAX StableHLO and optimized HLO, to identify observed optimizations such as fusion, layout handling, and GEMM lowering. Base performance conclusions on the measurements.

Include plots showing input/filter size variation, compilation times, the measurement methodology, correctness results, and evidence of compiler optimizations. Conclude with one paragraph comparing the four implementations across shapes. The generated `report_table.tex` contains measured data that can be included in the report. A detailed report outline is available in `COLAB_PART3.md`.


In [ ]:
# Download traces for a representative configuration, or download files individually from the sidebar.
for variant in ('eager', 'inductor', 'jax', 'cuda'):
    trace = Path(results_dir) / f'{variant}_h33_k5' / 'torch_trace.json'
    files.download(str(trace))
files.download(str(Path(results_dir) / 'jax_h33_k5' / 'jax_native.json.gz'))


## 7. Save results and sources before closing the runtime

The results archive excludes large compilation caches and preserves traces, plots, JSON data, IR, logs, and compilation diagnostics. If the report references Inductor-generated code, first copy the relevant cached `.py` files into a separate folder within the results directory.

Download the plots and measurements, complete Part III of the report, and submit according to the latest course hand-in instructions.


In [ ]:
results_zip = '/content/mp2_part3_results.zip'
sources_zip = '/content/mp2_part3_sources.zip'
subprocess.run(['zip', '-r', results_zip, results_dir, '-x', '*/cache/*'], check=True)
subprocess.run(['zip', '-r', sources_zip, '.', '-x', 'results_*/*', '*/__pycache__/*', '.git/*'], check=True)
files.download(results_zip)
files.download(sources_zip)
